# Лабораторная работа 7: параметрические эксперименты

В параметрической части исследуются:
- влияние интенсивности `λ` на характеристики очереди `M/M/c`;
- влияние числа машин `N` и числа ремонтников на среднее время до отказа
  в модели Росса.

In [1]:
using DrWatson
@quickactivate "project"

using CSV
using DataFrames

include(srcdir("DiscreteEventLab07Core.jl"))
using .DiscreteEventLab07

mkpath(datadir())
mkpath(plotsdir())

"/home/emkurilkoryumin/work/study/2026-1/2026-1==study--simulation-modeling/2026-1--study--simulation-modeling/labs/lab07/project/plots"

## Сканирование параметров `M/M/c`

Здесь варьируется интенсивность входного потока. Для каждой точки
сравниваются эмпирические показатели и аналитические формулы Эрланга `C`.

In [2]:
arrival_rates = [0.50, 0.65, 0.80, 0.90]
mmc_rows = DataFrame()

for (idx, arrival_rate) in enumerate(arrival_rates)
    result = simulate_mmc(
        num_customers = 2000,
        num_servers = 2,
        arrival_rate = arrival_rate,
        service_rate = 0.5,
        warmup_customers = 300,
        seed = 200 + idx,
    )
    append!(mmc_rows, result.summary)
end

CSV.write(datadir("mmc_scan.tsv"), mmc_rows; delim = '\t')

println("Параметрический сценарий M/M/c")
println(mmc_rows)

Параметрический сценарий M/M/c
4×15 DataFrame
 Row │ num_customers  warmup_customers  num_servers  arrival_rate  service_rate  mean_wait_sim  mean_wait_theory  mean_system_sim  mean_system_theory  avg_queue_sim  avg_queue_theory  utilization_sim  utilization_theory  prob_wait_sim  prob_wait_theory 
     │ Int64          Int64             Int64        Float64       Float64       Float64        Float64           Float64          Float64             Float64        Float64           Float64          Float64             Float64        Float64          
─────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │          2000               300            2          0.5            0.5       0.544634          0.666667          2.50273             2.66667       0.269346          0.333333         0

## Сканирование параметров модели Росса

Сначала используется полный вариант исследования по заданию:
меняются число машин `N` и число ремонтников, а среднее время до отказа
оценивается по серии независимых прогонов. Если такой режим окажется
слишком тяжёлым на конкретной машине, его можно упростить до сетки `2×2`
и `20` прогонов на точку.

In [3]:
machine_counts = [8, 10, 12, 14]
repairer_counts = [1, 2, 3]
num_runs = 30

ross_scan = ross_parameter_grid(
    machine_counts = machine_counts,
    repairer_counts = repairer_counts,
    num_runs = num_runs,
    S = 3,
    failure_mean = 20.0,
    repair_mean = 8.0,
    seed = 700,
)

CSV.write(datadir("ross_machine_scan.tsv"), ross_scan; delim = '\t')

println("Параметрический сценарий модели Росса")
println(ross_scan)

Параметрический сценарий модели Росса
12×9 DataFrame
 Row │ N      S      num_repairers  num_runs  crash_time_sim  crash_time_std  crash_time_theory  utilization  avg_queue_length 
     │ Int64  Int64  Int64          Int64     Float64         Float64         Float64            Float64      Float64          
─────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │     8      3              1        30        10.9725          5.62888           12.9083      0.72517           0.688624
   2 │    10      3              1        30         9.21833         4.61014            9.78125     0.729755          0.748405
   3 │    12      3              1        30         7.67642         3.87354            7.86808     0.714559          0.60505
   4 │    14      3              1        30         7.66257         4.62804            6.57883     0.728642          0.705833
   5 │     8      3              2        30        11.7